# Step 05: Training

Tests for `05-training/prepare_data.py` and `05-training/train.py`. Write code in the `.py` files, only import and test here.

Training itself is a script because it takes ~30 minutes. Run these in a terminal first:

```
uv run python 05-training/prepare_data.py   # once, a few seconds
uv run python 05-training/train.py          # about 30 minutes
```

Sections 1 to 5 test the pieces. Sections 6 to 8 read the results of the training run.

Run the cells top to bottom (Run All).

In [ ]:
%load_ext autoreload
%autoreload 2

import csv
import math
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import tiktoken

sys.path.append("05-training")
from train import (
    GPT, get_device, load_tokens, get_token_batch, get_lr, configure_optimizer, estimate_loss,
)

enc = tiktoken.get_encoding("gpt2")
device = get_device()
print("device:", device)

## 1. The data on disk

TinyStories, tokenized with GPT-2's BPE (the algorithm from step 01, pre-trained and fast) and saved as flat arrays of 16-bit ids. `np.memmap` reads from disk on demand, so the dataset never has to fit in RAM.

In [ ]:
train = load_tokens("train")
val = load_tokens("val")

print(f"train: {len(train):,} tokens, val: {len(val):,} tokens, dtype {train.dtype}")   # ~35.8M and ~0.36M, uint16
print(f"stories in train: {int((train == enc.eot_token).sum()):,}")                    # ~163,000 (each story ends with token 50256)
print(repr(enc.decode(train[:60].tolist())))

## 2. Batches

Same idea as step 02: random chunks, with `y` equal to `x` shifted one token to the right.

In [ ]:
np.random.seed(0)
x, y = get_token_batch(train, block_size=16, batch_size=4)

print(x.shape, y.shape)            # expect (4, 16) and (4, 16)
assert (x[:, 1:] == y[:, :-1]).all(), "y must be x shifted by one"
print(repr(enc.decode(x[0].tolist())))
print(repr(enc.decode(y[0].tolist())))   # same text, one token further

## 3. Learning-rate schedule

Warmup: the first steps use a tiny learning rate, because random weights give noisy gradients and big steps can wreck training. Then cosine decay: big steps early to make progress, small steps late to settle.

In [ ]:
max_lr, min_lr, warmup, total = 1e-3, 1e-4, 200, 4000
lrs = [get_lr(s, max_lr, min_lr, warmup, total) for s in range(total)]

for s in [0, 199, 200, 2100, 3999]:
    print(f"step {s:4d}: lr {lrs[s]:.2e}")
# step 0: 5.00e-06 | step 199: 1.00e-03 | step 200: 1.00e-03 | step 2100: 5.50e-04 | step 3999: ~1.00e-04

plt.plot(lrs)
plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("warmup then cosine decay")
plt.show()

## 4. Optimizer: weight decay on matrices only

Weight decay gently shrinks weights toward zero each step, which discourages memorizing. It is applied to weight matrices (2-D) but not to biases and LayerNorm gains (1-D).

In [ ]:
small = GPT(vocab_size=100, n_embd=32, n_heads=4, n_layers=2, block_size=16)
opt = configure_optimizer(small, lr=1e-3, weight_decay=0.1)

for g in opt.param_groups:
    print(f"weight_decay={g['weight_decay']}: {len(g['params'])} tensors, {sum(p.numel() for p in g['params']):,} values")
# expect: a decay group of matrices and a no-decay group of biases and LayerNorm gains

## 5. Gradient clipping

Occasionally one bad batch gives a huge gradient and one giant update ruins the weights. Clipping rescales the gradient so its total norm is at most 1.0. Normal batches are untouched.

In [ ]:
torch.manual_seed(0)
xb = torch.randint(0, 100, (8, 16))
yb = torch.randint(0, 100, (8, 16))
_, loss = small(xb, yb)
loss.backward()

before = torch.nn.utils.clip_grad_norm_(small.parameters(), max_norm=1e9)   # huge limit: just measures the norm
print("gradient norm:", round(before.item(), 3))

clipped = torch.nn.utils.clip_grad_norm_(small.parameters(), max_norm=0.1)  # pretend the limit is 0.1
after = torch.sqrt(sum(p.grad.pow(2).sum() for p in small.parameters()))
print("after clipping to 0.1:", round(after.item(), 3))   # expect 0.1 when the norm was above it

## 6. The training run

`train.py` logs to `05-training/out/log.csv`. Reading loss curves is the main skill in training:
- both falling: learning
- train falling, val flat or rising: memorizing (overfitting)
- spikes: unstable, the learning rate may be too high

In [ ]:
log_path = "05-training/out/log.csv"
assert os.path.exists(log_path), "run: uv run python 05-training/train.py"

rows = list(csv.DictReader(open(log_path)))
steps = [int(r["step"]) for r in rows]
tr = [float(r["train_loss"]) for r in rows]
va = [float(r["val_loss"]) for r in rows]
lr = [float(r["lr"]) for r in rows]

print(f"{len(rows)} evaluations, last step {steps[-1]}, {float(rows[-1]['elapsed_s']) / 60:.0f} min")
print(f"final train {tr[-1]:.3f} | final val {va[-1]:.3f}")   # expect about 2.15 and 2.13 (varies a little between runs)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(steps, tr, label="train"); ax[0].plot(steps, va, label="val")
ax[0].axhline(math.log(50257), color="gray", linestyle=":", label="random guess (ln 50257)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("loss")
ax[1].plot(steps, lr); ax[1].set_xlabel("step"); ax[1].set_title("learning rate")
plt.show()

## 7. What does the loss mean?

Loss is the average negative log-probability of the right next token. `exp(loss)` is the *perplexity*: roughly how many tokens the model is torn between at each step. Random guessing over the 50,257 tokens would be a perplexity of 50,257.

In [ ]:
print(f"start (random guess): perplexity {50257:,}")
print(f"final val loss {va[-1]:.3f}: perplexity {math.exp(va[-1]):.1f}")   # about 8: torn between ~8 tokens per step instead of 50,257

## 8. Load the checkpoint and write a story

The checkpoint stores the weights and the config, so we can rebuild the model. Step 06 improves how text is sampled. For now this uses plain sampling.

In [ ]:
ckpt = torch.load("05-training/out/ckpt.pt", map_location=device)
model = GPT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"{model.num_params():,} parameters, step {ckpt['step'] + 1}, lowest val loss seen in training {ckpt['best_val']:.3f}")

torch.manual_seed(0)
prompt = torch.tensor([enc.encode_ordinary("Once upon a time")], device=device)
out = model.generate(prompt, max_new_tokens=150)
print(enc.decode(out[0].tolist()))

## What is next

The model works, but generation is crude: pure sampling at temperature 1 can wander, and every new token recomputes the whole sequence. Step 06 adds temperature, top-k, top-p and the KV cache.